In [ ]:
# Projeto de Tratamento de Dados - Acidentes nas Rodovias Federais (PRF)

# Integrantes:
# Nome: Rafael Augusto Balko de Souza - RA: 2411551016
# Nome: Rafael Sanabria Zibordi - RA: 2411550905

In [ ]:
import pandas as pd
import folium
from folium.plugins import MiniMap, Fullscreen, Geocoder, MousePosition

print("Pandas versão:", pd.__version__)
print("Folium versão:", folium.__version__)

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [ ]:
# Carregando o dataset de acidentes de 2025
df_original = pd.read_csv('datatran2025.csv', sep=';', encoding='UTF-8')

# Exibindo o tamanho da base e as duas primeiras linhas
print(df_original.shape[0], 'linhas')
print(df_original.shape[1], 'colunas')
df_original.head(5)

In [ ]:
df_tratado = df_original.copy()

df_tratado['data_inversa'] = pd.to_datetime(df_tratado['data_inversa'], format='%d/%m/%Y', errors='coerce')
df_tratado[['data_inversa']].dtypes

In [ ]:
import pandas as pd

# 1. Gerando o ranking dos estados com mais acidentes
df_uf = df_tratado.value_counts(subset=['uf']) \
                  .to_frame().reset_index() \
                  .rename(columns={'count': 'total_acidentes'})

# 2. Apenas renomeando as colunas para o padrão limpo
df_uf = df_uf.rename(columns={
    'uf': 'UF',
    'total_acidentes': 'Total Acidentes'
})

# 3. Salvando a tabela gerada em um arquivo CSV independente
df_uf.to_csv('ocorrencias_por_estado_prf.csv', index=False)

# Exibindo o Top 10 na tela
df_uf.head(10)

In [ ]:
import folium
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, Geocoder, MousePosition

df_fatais = df_tratado.query('mortos > 0').copy()
df_fatais['data_inversa'] = pd.to_datetime(df_fatais['data_inversa'], format='%d/%m/%Y', errors='coerce')
df_fatais = df_fatais.dropna(subset=['data_inversa', 'latitude', 'longitude'])

df_fatais['latitude'] = df_fatais['latitude'].str.replace(',', '.').astype(float)
df_fatais['longitude'] = df_fatais['longitude'].str.replace(',', '.').astype(float)

print(f"Foram registrados {df_fatais.shape[0]} acidentes fatais em rodovias federais em 2025.")

# criando e separando o top 5 das principais causas dos acidentes
top_causas = df_fatais['causa_acidente'].value_counts().head(5).index.tolist()

# criando o mapa base centralizado no Brasil
mapa_prf = folium.Map(location=[-14.2350, -51.9253], zoom_start=4, tiles='OpenStreetMap')

MiniMap().add_to(mapa_prf)
Fullscreen().add_to(mapa_prf)
Geocoder().add_to(mapa_prf)
MousePosition().add_to(mapa_prf)

# criamos um dicionário para guardar um cluster dedicado a cada causa do Top 5 + as Outras Causas
camadas_causas = {}

for causa in top_causas:
    grupo = folium.FeatureGroup(name=f"{causa}").add_to(mapa_prf)
    camadas_causas[causa] = MarkerCluster(showCoverageOnHover=False).add_to(grupo)

# Criando uma camada extra para as causas menos frequentes
grupo_outras = folium.FeatureGroup(name="Causa: Outras Causas").add_to(mapa_prf)
camadas_causas['Outras'] = MarkerCluster(showCoverageOnHover=False).add_to(grupo_outras)

for index, linha in df_fatais.iterrows():
    
    br_info = int(linha['br']) if pd.notna(linha['br']) else "N/A"
    causa_atual = linha['causa_acidente']
    
    texto_popup = f"""
    <b>Data:</b> {linha['data_inversa'].strftime('%d/%m/%Y')}<br>
    <b>Local:</b> {linha['uf']} - BR {br_info} | KM {linha['km']}<br>
    <b>Município:</b> {linha['municipio']}<br>
    <b>Causa:</b> {causa_atual}<br>
    <b>Mortes:</b> <font color='red'><b>{linha['mortos']}</b></font>
    """
    
    cluster_destino = camadas_causas[causa_atual] if causa_atual in top_causas else camadas_causas['Outras']
    
    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']], 
        radius=5,
        popup=folium.Popup(texto_popup, max_width=300),
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7
    ).add_to(cluster_destino)

folium.LayerControl(collapsed=False).add_to(mapa_prf)

mapa_prf.save('mapa_acidentes_fatais.html')
mapa_prf

In [ ]:
# Criamos uma tabela que cruza o clima com a quantidade de mortos em cada acidente

analise_clima = pd.crosstab(
    df_tratado['condicao_metereologica'], 
    df_tratado['mortos'], 
    margins=True, 
    margins_name="Total de Acidentes"
).sort_values(by='Total de Acidentes', ascending=False)

# Ordenando pelo total de acidentes para ver o que mais aparece no topo
analise_clima = analise_clima.sort_values(by='Total de Acidentes', ascending=False)

analise_clima.index.name = "Condição Meteorológica"

analise_clima.to_csv('relatorio_clima.csv')
analise_clima

In [ ]:
import pandas as pd

# 1. Agrupando por Estado, Rodovia (BR) e KM para somar os mortos
ranking_km_perigosos = df_tratado.groupby(['uf', 'br', 'km']) \
                                 .agg(total_mortos=('mortos', 'sum'),
                                      total_acidentes=('id', 'count')) \
                                 .reset_index()

# 2. Pegando o Top 10 trechos com mais mortes
top_10_km_fatais = ranking_km_perigosos.nlargest(10, 'total_mortos')

# 3. Apenas renomeando as colunas para o padrão limpo
top_10_km_fatais = top_10_km_fatais.rename(columns={
    'uf': 'UF',
    'br': 'BR',
    'km': 'KM',
    'total_mortos': 'Total Mortos',
    'total_acidentes': 'Total Acidentes'
})

# Exibindo o Top 10 na tela com a numeração original do Pandas na esquerda
top_10_km_fatais

In [ ]:
!pip install plotly

In [ ]:
import plotly.express as px

# Agrupamos a coluna de dia da semana e quantidade de acidente
df_dias_semana = df_tratado.groupby(df_tratado['data_inversa'].dt.dayofweek).size().reset_index()
df_dias_semana.columns = ['Dia_Num', 'Quantidade_Acidentes']

# Mapeando os nomes dos dias
mapeamento_dias = {0: 'Segunda', 1: 'Terça', 2: 'Quarta', 3: 'Quinta', 4: 'Sexta', 5: 'Sábado', 6: 'Domingo'}
df_dias_semana['Dia_Semana'] = df_dias_semana['Dia_Num'].map(mapeamento_dias)

fig = px.line(
    df_dias_semana, 
    x='Dia_Semana', 
    y='Quantidade_Acidentes',
    title='<b>Acidentes por Dia da Semana (2025)</b>',
    labels={'Dia_Semana': 'Dia da Semana', 'Quantidade_Acidentes': 'Quantidade de Ocorrências'},
    markers=True
)

fig.update_traces(
    line_color='darkred', 
    line_width=3, 
    marker=dict(size=8),
    hovertemplate="<b>Dia da Semana:</b> %{x}<br><b>Quantidade de ocorrencia:</b> %{y:.3s}<extra></extra>"
)

fig.update_layout(
    title_font_size=16,
    xaxis_gridcolor='lightgray',
    yaxis_gridcolor='lightgray',
    plot_bgcolor='white'
)

# Exibe o gráfico interativo na tela
fig.show(renderer="notebook")

In [ ]:
import pandas as pd

# 1. Função para classificar o horário em turnos do dia
def definir_turno(horario_str):
    try:
        # Garante que o horário está no formato correto e pega apenas a hora
        hora = int(str(horario_str).split(':')[0])
        
        if 0 <= hora < 6:
            return 'Madrugada'
        elif 6 <= hora < 12:
            return 'Manhã'
        elif 12 <= hora < 18:
            return 'Tarde'
        else:
            return 'Noite'
    except:
        return 'Não Identificado'

# 2. Aplicando a função no nosso dataset para criar a nova coluna 'turno'
df_tratado['turno'] = df_tratado['horario'].apply(definir_turno)

# 3. Agrupando por turno para calcular o total de acidentes e o total de mortos
analise_turnos = df_tratado.groupby('turno').agg(
    Total_Acidentes=('id', 'count'),
    Total_Mortos=('mortos', 'sum')
).reset_index()

# 4. Ordenando pelos turnos que acumularam o maior número absoluto de mortos
analise_turnos = analise_turnos.sort_values(by='Total_Mortos', ascending=False)

# 5. Renomeando as colunas de dados para tirar o underline (_)
analise_turnos = analise_turnos.rename(columns={
    'Total_Acidentes': 'Total Acidentes',
    'Total_Mortos': 'Total Mortos'
})

# 6. Transformando a coluna 'turno' no índice e mudando para 'Turno' (com T maiúsculo)
analise_turnos = analise_turnos.set_index('turno')
analise_turnos.index.name = 'Turno'

# Exibe a tabela limpa e renomeada na tela
analise_turnos

In [ ]:
import pandas as pd

# 1. Agrupando pelo tipo do acidente para extrair o volume de ocorrências e dados de óbitos
analise_tipos = df_tratado.groupby('tipo_acidente').agg(
    Total_Acidentes=('id', 'count'),
    Total_Mortos=('mortos', 'sum')
).reset_index()

# 2. Filtrando para exibir apenas os tipos de acidentes que possuem um histórico relevante (mais de 10 ocorrências)
analise_tipos = analise_tipos.query('Total_Acidentes > 10')

# 3. Ordenando a tabela pelo total de mortos para colocar o tipo mais fatal no topo
analise_tipos = analise_tipos.sort_values(by='Total_Mortos', ascending=False)

# 4. Renomeando as colunas de dados para tirar o underline (_)
analise_tipos = analise_tipos.rename(columns={
    'Total_Acidentes': 'Total Acidentes',
    'Total_Mortos': 'Total Mortos'
})

# 5. Transforma a coluna 'tipo_acidente' no índice e muda o nome para 'Tipo Acidente'
analise_tipos = analise_tipos.set_index('tipo_acidente')
analise_tipos.index.name = 'Tipo Acidente'

# Exibe o ranking definitivo de violência por tipo de batida
analise_tipos

In [ ]:
import pandas as pd

# 1. Gerando o ranking das causas de acidentes
df_causas = df_tratado.value_counts(subset=['causa_acidente']) \
                      .to_frame().reset_index() \
                      .rename(columns={'count': 'total_ocorrencias'})

# 2. Apenas renomeando as colunas para o padrão limpo
df_causas = df_causas.rename(columns={
    'causa_acidente': 'Causa Acidente',
    'total_ocorrencias': 'Total Ocorrências'
})

# 3. Salvando a tabela gerada em um arquivo CSV independente
df_causas.to_csv('causas_acidentes_prf.csv', index=False)

# Exibindo o Top 10 na tela
df_causas.head(10)